In [0]:
# Parameters (all optional). Leave public_dir blank to derive it from this
# notebook's location + relative_public_path, so the notebook works in any clone.
dbutils.widgets.text("public_dir", "", "Output public/ dir (blank = derive)")
dbutils.widgets.text("relative_public_path", "../../client/public", "public/ relative to this notebook")
dbutils.widgets.text("brand_subdir", "brand", "Subdir for symbol SVGs")
dbutils.widgets.text("assets_csv", "/Workspace/Shared/brandfolder/assets.csv", "Brandfolder assets catalog")

In [0]:
# Download Databricks brand assets (symbols and favicons) from the brandfolder
# CDN into an app's public/ dir (default: ../../client/public from here).
#
# Run once on compute that has outbound internet access.
# After running, commit the downloaded files.
#
# Output location and catalog come from the Parameters cell widgets.
# CDN URLs are looked up by filename in the assets catalog (assets_csv).
# Plan reference: docs/plans/ui_brand_plan.md §4 (Assets/identity)

import urllib.request, os, hashlib, csv
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from PIL import Image  # pillow is pre-installed on DBR

# --------------------------------------------------------------------------- #
# Configuration (from widgets; nothing user- or repo-specific is hard-coded)
# --------------------------------------------------------------------------- #
def _notebook_dir() -> Path:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    nb_path = ctx.notebookPath().get()
    if not nb_path.startswith("/Workspace"):
        nb_path = "/Workspace" + nb_path
    return Path(nb_path).parent

public_dir = dbutils.widgets.get("public_dir").strip()
if public_dir:
    APP_PUBLIC = Path(public_dir)
else:
    rel = dbutils.widgets.get("relative_public_path").strip()
    APP_PUBLIC = Path(os.path.normpath(_notebook_dir() / rel))

if not APP_PUBLIC.is_dir():
    raise FileNotFoundError(
        f"Output dir {APP_PUBLIC} does not exist. Set the public_dir widget "
        "or fix relative_public_path (this notebook won't create it, to avoid "
        "writing assets to the wrong place)."
    )
BRAND_DIR = APP_PUBLIC / dbutils.widgets.get("brand_subdir").strip()
ASSETS_CSV = Path(dbutils.widgets.get("assets_csv").strip())
print(f"Output: {APP_PUBLIC}\nBrand subdir: {BRAND_DIR}\nCatalog: {ASSETS_CSV}\n")

# Look up CDN URLs by filename in the brandfolder catalog. The catalog URLs
# carry ?auto=webp&format=png, which converts SVGs to PNG; strip the query to
# get the original file.
with open(ASSETS_CSV, newline="") as f:
    _catalog = {r["Filename"]: (r.get("Attachment CDN URL") or r.get("File URL") or "")
                for r in csv.DictReader(f) if r.get("Filename")}

def cdn_url(filename: str) -> str:
    url = _catalog.get(filename)
    if not url:
        raise KeyError(f"{filename} not found in {ASSETS_CSV}")
    parts = urlsplit(url)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))

SVG_FILES = ["databricks-symbol-color.svg", "databricks-symbol-light.svg"]
FAVICON_FILES = [
    "databricks-favicon-16x16-orange.png",
    "databricks-favicon-32x32-orange.png",
    "databricks-favicon-48x48-orange.png",
    "databricks-favicon-512x512-orange.png",
]
SVG_ASSETS = {f: cdn_url(f) for f in SVG_FILES}
FAVICON_ASSETS = {f: cdn_url(f) for f in FAVICON_FILES}

# --------------------------------------------------------------------------- #
# Download helpers
# --------------------------------------------------------------------------- #
def download(url: str, dest: Path) -> None:
    """Download url to dest, creating parent dirs. Prints size + SHA-256."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, str(dest))
    size = dest.stat().st_size
    sha = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
    print(f"  {dest.name:.<50} {size:>8,} bytes  sha256:{sha}")

# --------------------------------------------------------------------------- #
# 1. Download SVGs into brand/
# --------------------------------------------------------------------------- #
print("=== SVG symbols (brand/) ===")
for name, url in SVG_ASSETS.items():
    download(url, BRAND_DIR / name)

# --------------------------------------------------------------------------- #
# 2. Download official favicon PNGs from brandfolder
#    The 16, 32 and 48 px icons are hand-tuned by the brand team and sharper
#    than resizing the 512 px source. We use them directly as the app favicons.
# --------------------------------------------------------------------------- #
print("\n=== Favicon PNGs (official) ===")
# Map brandfolder filenames → the names index.html expects
OFFICIAL_FAVICONS = {
    "databricks-favicon-16x16-orange.png":  "favicon-16x16.png",
    "databricks-favicon-32x32-orange.png":  "favicon-32x32.png",
    "databricks-favicon-48x48-orange.png":  "favicon-48x48.png",
    "databricks-favicon-512x512-orange.png": "favicon-512x512.png",
}
for src_name, dest_name in OFFICIAL_FAVICONS.items():
    url = FAVICON_ASSETS[src_name]
    download(url, APP_PUBLIC / dest_name)

# --------------------------------------------------------------------------- #
# 3. Derive sizes that brandfolder doesn't supply
#    Only 192 px (PWA manifest) and 180 px (Apple touch icon) need generation.
# --------------------------------------------------------------------------- #
print("\n=== Derived favicons ===")
img = Image.open(APP_PUBLIC / "favicon-512x512.png")

DERIVED = {
    "favicon-192x192.png": 192,
    "apple-touch-icon.png": 180,
}
for name, size in DERIVED.items():
    dest = APP_PUBLIC / name
    resized = img.resize((size, size), Image.LANCZOS)
    resized.save(str(dest), "PNG")
    sha = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
    print(f"  {name:.<50} {size}x{size}  sha256:{sha}")

# --------------------------------------------------------------------------- #
# 4. Copy symbol-color as favicon.svg (replacing the old AppKit placeholder)
# --------------------------------------------------------------------------- #
print("\n=== favicon.svg ===")
import shutil
fav_svg = APP_PUBLIC / "favicon.svg"
shutil.copy2(BRAND_DIR / "databricks-symbol-color.svg", fav_svg)
print(f"  favicon.svg ← {(BRAND_DIR / 'databricks-symbol-color.svg').relative_to(APP_PUBLIC)} ({fav_svg.stat().st_size:,} bytes)")

# --------------------------------------------------------------------------- #
# 5. Summary
# --------------------------------------------------------------------------- #
print("\n=== Final public/ brand files ===")
for p in sorted(APP_PUBLIC.rglob("*")):
    if p.is_file() and any(k in p.name.lower() for k in ["favicon", "symbol", "apple-touch", "databricks"]):
        print(f"  {p.relative_to(APP_PUBLIC)}  ({p.stat().st_size:,} bytes)")

print("\n✓ Done. Run the next cell to verify no placeholders remain, then commit.")
print(f"  Next: git add -A {APP_PUBLIC} && git diff --cached --stat")

In [0]:
# Verify that no placeholder or scaffold brand assets remain in public/.
# This cell can also be run independently as a gate check.
#
# Checks:
#   1. SVGs in brand/ must NOT contain the word "Placeholder"
#   2. No file may contain the old AppKit Lava 300 scaffold colour #FABFBA
#   3. favicon.svg must contain the Databricks brand colour #FF3621
#   4. favicon PNGs must be valid images at their expected sizes
#   5. Symbol SVGs must be valid SVG (start with <svg or <?xml)

import re
from pathlib import Path
from PIL import Image

errors: list[str] = []

# Re-derive APP_PUBLIC the same way cell 2 does, so this cell works standalone
try:
    _pub = APP_PUBLIC
except NameError:
    import os
    def _nb_dir() -> Path:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb = ctx.notebookPath().get()
        if not nb.startswith("/Workspace"):
            nb = "/Workspace" + nb
        return Path(nb).parent
    _rel = dbutils.widgets.get("relative_public_path").strip() or "../../client/public"
    _pub = Path(os.path.normpath(_nb_dir() / _rel))
    _brand_sub = dbutils.widgets.get("brand_subdir").strip() or "brand"

_brand = _pub / (_brand_sub if "_brand_sub" in dir() else dbutils.widgets.get("brand_subdir").strip())

# --- Check 1: No "Placeholder" text in SVGs ---
for svg in _brand.glob("*.svg"):
    text = svg.read_text()
    if "Placeholder" in text:
        errors.append(f"PLACEHOLDER TEXT in {svg.relative_to(_pub)}")

# --- Check 2: No AppKit scaffold colour #FABFBA anywhere ---
for f in _pub.rglob("*.svg"):
    if "FABFBA" in f.read_text().upper():
        errors.append(f"SCAFFOLD COLOUR #FABFBA in {f.relative_to(_pub)}")

# --- Check 3: favicon.svg contains Databricks brand colour ---
fav_svg = _pub / "favicon.svg"
if fav_svg.exists():
    content = fav_svg.read_text()
    if not content.lstrip().startswith(("<svg", "<?xml")):
        errors.append("favicon.svg is not a valid SVG")
    if "FF3621" not in content.upper():
        errors.append("favicon.svg missing Databricks brand colour #FF3621")
else:
    errors.append("favicon.svg does not exist")

# --- Check 4: Favicon PNGs are valid images at expected sizes ---
EXPECTED_PNGS = {
    "favicon-16x16.png": (16, 16),
    "favicon-32x32.png": (32, 32),
    "favicon-48x48.png": (48, 48),
    "favicon-192x192.png": (192, 192),
    "favicon-512x512.png": (512, 512),
    "apple-touch-icon.png": (180, 180),
}
for name, expected_size in EXPECTED_PNGS.items():
    p = _pub / name
    if not p.exists():
        errors.append(f"{name} missing")
        continue
    try:
        im = Image.open(p)
        if im.size != expected_size:
            errors.append(f"{name} is {im.size}, expected {expected_size}")
    except Exception as e:
        errors.append(f"{name} not a valid image: {e}")

# --- Check 5: Symbol SVGs are valid ---
for name in ["databricks-symbol-color.svg", "databricks-symbol-light.svg"]:
    p = _brand / name
    if not p.exists():
        errors.append(f"brand/{name} missing")
        continue
    text = p.read_text().lstrip()
    if not text.startswith(("<svg", "<?xml")):
        errors.append(f"brand/{name} is not a valid SVG")

# --- Report ---
if errors:
    print("FAIL — placeholder or scaffold brand assets detected:\n")
    for e in errors:
        print(f"  ✘ {e}")
    raise AssertionError(f"{len(errors)} brand asset check(s) failed")
else:
    print("✓ All brand asset checks passed:")
    print("  • No 'Placeholder' text in SVGs")
    print("  • No scaffold colour #FABFBA")
    print("  • favicon.svg is a valid SVG with #FF3621")
    print(f"  • {len(EXPECTED_PNGS)} favicon PNGs at correct sizes")
    print("  • Symbol SVGs are valid")